In [1]:
!pip install -U google-genai

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from google import genai
from google.genai import types
import json
import re

with open("api_key.txt", "r") as f:
    GEMINI_API_KEY = f.read().strip()

client = genai.Client(api_key=GEMINI_API_KEY)

LLM_MODEL = "gemini-3.5-flash-lite"

print("Gemini ready")

Gemini ready


In [3]:
document = """
PostgreSQL is an open-source relational database management system.
It is widely used for transactional and analytical applications.

PostgreSQL supports ACID transactions, constraints, indexes, views,
stored procedures, and many other database capabilities.

PostgreSQL can be extended using extensions. Extensions add additional
functionality without modifying the PostgreSQL core system.

pgvector is a PostgreSQL extension designed for working with vector
embeddings. It allows applications to store vectors inside PostgreSQL
and perform similarity searches.

Embeddings are numerical representations of text. Text with similar
meaning can have vectors that are close together in vector space.

Vector similarity search is important for Retrieval Augmented Generation.
A RAG system converts documents and user questions into embeddings and
retrieves document chunks that are semantically similar to the question.

HNSW is an approximate nearest-neighbor indexing method supported by
pgvector. It can make vector similarity searches much faster when the
database contains a large number of vectors.

Redis is an in-memory data store commonly used for caching. Applications
can use Redis to reduce repeated database queries and improve response
times.

Redis can also be used for session storage, rate limiting, queues, and
other high-speed data access scenarios.
"""

print(document)


PostgreSQL is an open-source relational database management system.
It is widely used for transactional and analytical applications.

PostgreSQL supports ACID transactions, constraints, indexes, views,
stored procedures, and many other database capabilities.

PostgreSQL can be extended using extensions. Extensions add additional
functionality without modifying the PostgreSQL core system.

pgvector is a PostgreSQL extension designed for working with vector
embeddings. It allows applications to store vectors inside PostgreSQL
and perform similarity searches.

Embeddings are numerical representations of text. Text with similar
meaning can have vectors that are close together in vector space.

Vector similarity search is important for Retrieval Augmented Generation.
A RAG system converts documents and user questions into embeddings and
retrieves document chunks that are semantically similar to the question.

HNSW is an approximate nearest-neighbor indexing method supported by
pgvector. It

In [4]:
paragraphs = [
    p.strip()
    for p in document.split("\n\n")
    if p.strip()
]

for i, paragraph in enumerate(paragraphs):
    print(f"\n--- Paragraph {i} ---")
    print(paragraph)


--- Paragraph 0 ---
PostgreSQL is an open-source relational database management system.
It is widely used for transactional and analytical applications.

--- Paragraph 1 ---
PostgreSQL supports ACID transactions, constraints, indexes, views,
stored procedures, and many other database capabilities.

--- Paragraph 2 ---
PostgreSQL can be extended using extensions. Extensions add additional
functionality without modifying the PostgreSQL core system.

--- Paragraph 3 ---
pgvector is a PostgreSQL extension designed for working with vector
embeddings. It allows applications to store vectors inside PostgreSQL
and perform similarity searches.

--- Paragraph 4 ---
Embeddings are numerical representations of text. Text with similar
meaning can have vectors that are close together in vector space.

--- Paragraph 5 ---
Vector similarity search is important for Retrieval Augmented Generation.
A RAG system converts documents and user questions into embeddings and
retrieves document chunks that are 

In [5]:
paragraph_text = "\n\n".join(
    f"PARAGRAPH {i}:\n{p}"
    for i, p in enumerate(paragraphs)
)

prompt = f"""
You are an expert document chunking agent.

Your task is to divide the following document paragraphs into
meaningful chunks for a Retrieval Augmented Generation (RAG) system.

Rules:

1. Keep paragraphs together when they discuss the same concept.
2. Separate paragraphs when the main topic changes.
3. Do not split a paragraph.
4. A chunk should contain enough context to answer a question.
5. Avoid creating unnecessarily small chunks.
6. Each paragraph must belong to exactly one chunk.
7. Preserve the original paragraph order.

Return ONLY valid JSON.

Use this format:

{{
  "chunks": [
    {{
      "chunk_id": 0,
      "paragraph_ids": [0, 1],
      "topic": "short topic name",
      "reason": "why these paragraphs belong together"
    }}
  ]
}}

DOCUMENT:

{paragraph_text}
"""

response = client.models.generate_content(
    model=LLM_MODEL,
    contents=prompt,
    config=types.GenerateContentConfig(
        temperature=0
    )
)

print(response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{
  "chunks": [
    {
      "chunk_id": 0,
      "paragraph_ids": [0, 1],
      "topic": "PostgreSQL Overview",
      "reason": "Both paragraphs introduce PostgreSQL as an open-source relational database and describe its general features and transactional capabilities."
    },
    {
      "chunk_id": 1,
      "paragraph_ids": [2, 3],
      "topic": "PostgreSQL Extensions and pgvector",
      "reason": "These paragraphs discuss PostgreSQL extensions, specifically introducing the pgvector extension for storing vector embeddings and performing similarity searches."
    },
    {
      "chunk_id": 2,
      "paragraph_ids": [4, 5, 6],
      "topic": "Embeddings, RAG, and HNSW Indexing",
      "reason": "These paragraphs explain vector embeddings, their role in Retrieval Augmented Generation (RAG) similarity searches, and how the HNSW indexing method accelerates these searches in pgvector."
    },
    {
      "chunk_id": 3,
      "paragraph_ids": [7, 8],
      "topic": "Redis Data Store",
   

In [6]:
raw_response = response.text.strip()

# Remove markdown code fences if Gemini adds them
raw_response = re.sub(
    r"^```json\s*|\s*```$",
    "",
    raw_response,
    flags=re.IGNORECASE
)

chunk_plan = json.loads(raw_response)

print(json.dumps(chunk_plan, indent=2))

{
  "chunks": [
    {
      "chunk_id": 0,
      "paragraph_ids": [
        0,
        1
      ],
      "topic": "PostgreSQL Overview",
      "reason": "Both paragraphs introduce PostgreSQL as an open-source relational database and describe its general features and transactional capabilities."
    },
    {
      "chunk_id": 1,
      "paragraph_ids": [
        2,
        3
      ],
      "topic": "PostgreSQL Extensions and pgvector",
      "reason": "These paragraphs discuss PostgreSQL extensions, specifically introducing the pgvector extension for storing vector embeddings and performing similarity searches."
    },
    {
      "chunk_id": 2,
      "paragraph_ids": [
        4,
        5,
        6
      ],
      "topic": "Embeddings, RAG, and HNSW Indexing",
      "reason": "These paragraphs explain vector embeddings, their role in Retrieval Augmented Generation (RAG) similarity searches, and how the HNSW indexing method accelerates these searches in pgvector."
    },
    {
      "chu

In [7]:
def validate_chunk_plan(chunk_plan, paragraph_count):

    chunks = chunk_plan["chunks"]

    all_ids = []

    for chunk in chunks:
        all_ids.extend(chunk["paragraph_ids"])

    expected = list(range(paragraph_count))

    if sorted(all_ids) != expected:
        raise ValueError(
            f"Invalid chunk plan.\n"
            f"Expected: {expected}\n"
            f"Received: {sorted(all_ids)}"
        )

    print("Chunk plan is valid.")

In [8]:
validate_chunk_plan(
    chunk_plan,
    len(paragraphs)
)

Chunk plan is valid.


In [9]:
final_chunks = []

for chunk in chunk_plan["chunks"]:

    content = "\n\n".join(
        paragraphs[i]
        for i in chunk["paragraph_ids"]
    )

    final_chunks.append({
        "chunk_id": chunk["chunk_id"],
        "topic": chunk["topic"],
        "reason": chunk["reason"],
        "content": content
    })

In [10]:
for chunk in final_chunks:

    print("=" * 80)
    print("CHUNK:", chunk["chunk_id"])
    print("TOPIC:", chunk["topic"])
    print("REASON:", chunk["reason"])
    print()
    print(chunk["content"])

CHUNK: 0
TOPIC: PostgreSQL Overview
REASON: Both paragraphs introduce PostgreSQL as an open-source relational database and describe its general features and transactional capabilities.

PostgreSQL is an open-source relational database management system.
It is widely used for transactional and analytical applications.

PostgreSQL supports ACID transactions, constraints, indexes, views,
stored procedures, and many other database capabilities.
CHUNK: 1
TOPIC: PostgreSQL Extensions and pgvector
REASON: These paragraphs discuss PostgreSQL extensions, specifically introducing the pgvector extension for storing vector embeddings and performing similarity searches.

PostgreSQL can be extended using extensions. Extensions add additional
functionality without modifying the PostgreSQL core system.

pgvector is a PostgreSQL extension designed for working with vector
embeddings. It allows applications to store vectors inside PostgreSQL
and perform similarity searches.
CHUNK: 2
TOPIC: Embeddings, RA